In [ ]:
# import 

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import pyplot
import os
import matplotlib.ticker as ticker
import numpy as np
import xarray as xr
import sys
from utilities import find_best_grid_point, get_station_coords,form_xdate, get_anomalies
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from plotting import tol_colors # color schemes from https://personal.sron.nl/~pault/

#activate interactive figures
%matplotlib widget
#activate autoreload
%load_ext autoreload

## Add parent directory to syspath
parent_dir = os.path.abspath(os.path.join(os.path.dirname('.'), '..'))
if not parent_dir in sys.path:
    sys.path.append(parent_dir)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
## Read all data
%autoreload 2
from analyses_level2.read_data import AvailableData, create_data_reader

# File path
data_path = "../data/"

# if New data is added to ./data folder, adapt the dictionary in AvailableData
all_data = list(AvailableData)
print(all_data)

#####---------- TO ADAPT ---------------#####
selected_data = ['CO2', 'CO2_flask', 
                 'CO', 'CO_flask', 
                 'CH4', 'CH4_flask', 
                 'O3'
                 ] # define data to read in. If empty, all data is used 
## 
processing_kwargs = { 
    'FLASK_FLAG_CORR' : True # exclude flagged flask-data
}
#####-----------------------------------#####

datasets = [] # initialize list of all datasets 
# read in data
for sel in (selected_data if selected_data else all_data):
    #define where the data has to be read from
    data_reader =  create_data_reader(data_path=data_path,dataset=sel,**processing_kwargs) #creates an instance of the desired data_reader class
    print(f"Data from {data_reader.__class__.__name__} for {sel}:")

    # call the data-reading function on that instance: 
    data = data_reader.read_data() 
    # call the data-processing
    data = data_reader.process_data(data)

    # prepare merged dataset
    data = data.drop(columns='endtime') # problem when merging datasets (because of NaT?), so better remove endtime
    ds = data.to_xarray()
    ds = ds.assign_coords(dataset=sel)
    ds['species'] = data_reader.species
    ds['unit']  = np.unique(ds.unit.dropna(dim='time'))[0]
    datasets.append(ds)

# save all in one xarray dataset
ds_all = xr.concat(datasets,dim="dataset")


In [ ]:
ds_all

In [ ]:
## check flask data
sel_spec = 'CO2_flask'

plt.figure()
ds = ds_all.sel(dataset=sel_spec)
ds = ds.where(~np.isnan(ds.value), drop=True) # remove times where we have no data
ds.QCflag.plot(ls='',marker='.')
plt.show()


In [ ]:
#select all flask species and plot flags: (set 'FLASK_FLAG_CORR' : False)
sel_species = [s for s in selected_data if 'flask' in s]

fig, axs = plt.subplots(len(sel_species),1,sharex=True)
for s,ax in zip(sel_species,axs):
    ds = ds_all.sel(dataset=s)
    ds.value.plot(ax=ax,ls='',marker='.')
    
    # check QCflag = 3
    ds.where(ds.QCflag==3).value.plot(ax=ax,ls='',marker='.',c='r')

    #check original flag (reject if first character is not '.' ) -> gives the same!
    #mask_flag = [str(s)[0] != '.' if isinstance(s, str) else False for s in ds.ORG_QCflag.values]
    #mask_dataarray = xr.DataArray(mask_flag, dims='time', coords={'time': ds['time']})
    #if any(mask_flag):
    #    ds.where(mask_dataarray,drop=True).value.plot(ax=ax,ls='',marker='x',c='g')


In [ ]:
sel_species = ds_all.species 
unique_species = np.unique(sel_species)

#moving window
mw = 24*3 #hours

fig, axs = plt.subplots(len(unique_species),1,sharex=True,figsize=(8,len(unique_species)*2))
for s,ax in zip(unique_species,axs):
    ds_sel = ds_all.where(ds_all.species==s,drop=True)

    #function to plot each subplot
    def plot_data(i_s,mw_temp):
        ds_sel.sel(dataset=i_s).value.plot(ax=ax,ls='',marker='.',alpha=0.7,label=str(i_s))
        #moving mean
        if ii.astype(str).str.contains('flask') == False:
            ds_sel.sel(dataset=i_s).value.rolling(time=mw_temp,center=True,min_periods=mw_temp/2).mean().plot(
            ax=ax,ls='-',c='k',label=f'Moving Mean ({int(mw_temp/24)}days)') #

    if len(ds_sel.dataset)>1: #several datasets with same species
        for ii in ds_sel.dataset:
            # plot data (adapt moving window for flask)
            plot_data(ii.values, mw) #mw*3 if ii.astype(str).str.contains('flask') else mw
    else:
        plot_data(s,mw)
    
    ax.set_ylabel(f'{ds_sel.species.values[0]} ({ds_sel.unit.values[0]})')
    ax.set_xlabel('')
    ax.set_title('')
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper left')
plt.suptitle('Mt. Kenya GAW station')
plt.tight_layout()

    


In [ ]:
#moving window
mw = 24*3 #hours

plt.figure()
ds = ds_all.sel(dataset='CO2')
ds.value.plot(ls='',marker='.')

ds.value.dropna('time').rolling(time=mw,center=True,min_periods=mw/2).mean().plot(ls='-')

# check QCflag = 3
ds.where(ds.QCflag==3).value.plot(ls='',marker='.',c='r')


plt.show()

### Compare to CAMS

In [ ]:

%autoreload 2
import read_cams

## this takes some time (need to do it once, then just load the nc file in the next cell)
#cams_best_grid = read_cams.get_best_cams(ds_all,save_netcdf=True)

In [ ]:
cams_best_grid = xr.open_dataset(f"{data_path}/cams/cams_best_grid_MKN.nc")

In [ ]:
## resample observations
# remove unnecessary variables for simplication
variables_to_keep= ['time','dataset','value','value_unc','unit','species']
ds_all_simple = ds_all.drop_vars(set(ds_all.variables) - set(variables_to_keep))


# Resample only time-dependent variables
time_dependent_variables = [var for var in ds_all_simple.data_vars if 'time' in ds_all_simple[var].dims]
non_time_dependent_variables = [var for var in ds_all_simple.data_vars if var not in time_dependent_variables or not 'time']

ds_all_3h = ds_all_simple[time_dependent_variables].resample(time='3h').mean(keep_attrs=True)
ds_all_6h = ds_all_simple[time_dependent_variables].resample(time='6h').mean(keep_attrs=True)
# add non-time dependent variables again
for var in non_time_dependent_variables:
    ds_all_3h[var] = ds_all_simple[var]
    ds_all_6h[var] = ds_all_simple[var]

##### Check the selected CAMS grids

In [ ]:
# Read all full cams datasets
dir_data_cams = r"../data/cams"
cams_invgg_co2 = xr.open_dataset(
    dir_data_cams + r"/cams_invGG_co2_2020_2023_MKN.nc"
)
cams_invgg_ch4 = xr.open_dataset(
    dir_data_cams + r"/cams_invGG_ch4_2020_2021_MKN.nc"
)
cams_eac4 = xr.open_dataset(dir_data_cams + r"/cams_eac4_2003_2022_MKN.nc")
cams_egg4 = xr.open_dataset(dir_data_cams + r"/cams_egg4_2003_2020_MKN.nc")

cams_gfas = xr.open_dataset(dir_data_cams + r"/cams_gfas_2020_2023.nc")

In [ ]:
## Show which grids are selcted in cams data
#mkn station: 
lat,lon,alt= get_station_coords('MKN')

fig, axs = plt.subplots(3,2,figsize=(10,8))
for ds_name, ax in zip(cams_best_grid.dataset,axs.reshape(-1)):
#ds_name = 'co2_invgg'

    if ds_name == "co2_invgg":
        cams_sel = cams_invgg_co2["CO2"]
    elif ds_name == "co2_egg4":
        cams_sel = cams_egg4["CO2"]
    #CH4
    elif ds_name == "ch4_invgg":
        cams_sel = cams_invgg_ch4["CH4"]
    elif ds_name == "ch4_egg4":
        cams_sel = cams_egg4["CH4"] 
    #CO
    elif ds_name == "co_eac4":
        cams_sel = cams_eac4["CO"]
    #O3
    elif ds_name == "o3_eac4":
        cams_sel = cams_eac4["O3"]

    # selected grid: 
    cams_lev = cams_best_grid.sel(dataset=ds_name).level.values
    cams_lat = cams_best_grid.sel(dataset=ds_name).latitude.values
    cams_lon = cams_best_grid.sel(dataset=ds_name).longitude.values

    
    cams_sel.sel(level=cams_lev).isel(time=0).plot(ax=ax)
    ax.plot(lon, lat, marker="x",color='w')  # mt. kenya station
    # Draw a red rectangle around the selected grid cell
    dy = np.unique(np.diff(cams_sel.latitude))
    dx = np.unique(np.diff(cams_sel.longitude))
    lat_min, lat_max = cams_lat - dy / 2, cams_lat + dy / 2
    lon_min, lon_max = cams_lon - dx / 2, cams_lon + dx / 2
    ax.plot(
        [lon_min, lon_max, lon_max, lon_min, lon_min],
        [lat_min, lat_min, lat_max, lat_max, lat_min],
        color="red",
    )
    ax.set_title(f'{ds_name.values}, level={str(cams_lev)}')
plt.tight_layout()
plt.show()


In [ ]:
###### --- FOR ALL comparison FIGURES --- #####
## Select the time period and species to plot


tsel1 = "2020-01-01"
tsel2 = "2023-12-31"

sel_species = ds_all.species
unique_species = np.unique(sel_species)

species_sel = unique_species

#species_sel = ["CO2", "CO"]

In [ ]:
## plot cams versus observations
alpha = 0.7
ms = 8

fig, axs = plt.subplots(
    len(species_sel), 1, sharex=True, figsize=(10, len(species_sel) * 2)
)
for s, ax in zip(species_sel, axs):
    if s == "CH4":
        ds = ds_all_6h  # ch4 has 6h resolution
    else:
        ds = ds_all_3h
    
    #OR: use hourly observations
    #ds = ds_all

    ds_sel = ds.sel(time=slice(tsel1, tsel2)).where(ds.species == s, drop=True)
    # select cams datasets that exist for this species
    cams_datasets_sel = [cams_sel for cams_sel in cams_best_grid.dataset.values if f"{s.lower()}_" in cams_sel]
    cams = cams_best_grid.sel(dataset=cams_datasets_sel, time=slice(tsel1, tsel2))

    # function to plot each subplot
    def plot_data(ds,label):
        if 'flask' in str(ds.dataset.values):
            m = '.'
            l = ''
        else:
            m = '.'
            l = '-'
        ds.plot(
            ax=ax,
            ls=l,
            marker=m,
            markeredgewidth=0,
            alpha=alpha,
            markersize=ms,
            label=f"{label} {str(ds.dataset.values)}",
        )

    # plot obs
    if len(ds_sel.dataset) > 1:  # several obs datasets with same species
        # plot only if we have data in our given time period:
        variables_to_keep = [
                spec.dataset.values
                for spec in ds_sel.dataset
                if ds_sel.sel(dataset=spec)["value"].isnull().all() == False
            ]
        for ds_i in ds_sel.sel(dataset=variables_to_keep).dataset:
            plot_data(ds_sel.sel(dataset=ds_i.values)["value"], 'obs')
    else:
        plot_data(ds_sel.sel(dataset=s)["value"], 'obs')

    # plot cams
    if len(cams.dataset) > 1:  # several cams datasets with same species
        # plot only if we have data in our given time period:
        variables_to_keep = [
                spec.dataset.values
                for spec in cams.dataset
                if cams.sel(dataset=spec)["value"].isnull().all() == False
            ]
        for cams_i in cams.sel(dataset=variables_to_keep).dataset:
            plot_data(cams.sel(dataset=cams_i)["value"], 'cams')
    else:
        plot_data(cams.isel(dataset=0)["value"], 'cams')

    ax.set_ylabel(f"{ds_sel.species[0].values} ({ds_sel.unit[0].values})")
    ax.set_xlabel("")
    ax.set_title("")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
## plot bias between cams and observations
def root_mean_squared_error(x, y):
    return np.sqrt(((x - y) ** 2).mean(dim="time"))


def mean_bias(x, y):
    return (x).mean(dim="time") - (y).mean(dim="time")


alpha = 0.7
ms = 8

fig, axs = plt.subplots(
    len(species_sel), 1, sharex=True, figsize=(10, len(species_sel) * 2)
)
for s, ax in zip(species_sel, axs):
    # select CAMS for which I want to show the bias
    if s == "CO2":
        cams_dataset = "co2_invgg" #"co2_egg4" 
    elif s == "CH4":
        cams_dataset = "ch4_invgg" #"ch4_egg4"
    elif s == "CO":
        cams_dataset = "co_eac4"
    elif s == "O3":
        cams_dataset = "o3_eac4"

    # select obs, ch4_invgg has 6h resolution
    if cams_dataset == "ch4_invgg": 
        ds = ds_all_6h  # ch4 has 6h resolution
    else:
        ds = ds_all_3h

    ds_sel = ds.sel(time=slice(tsel1, tsel2)).where(ds.species == s, drop=True)

    # function to plot each subplot
    def plot_bias(x):
        x.plot(
            ax=ax,
            ls="-",
            marker=".",
            markeredgewidth=0,
            alpha=alpha,
            markersize=ms,
            label=f"cams-obs",
        )

    # plot bias
    cams = cams_best_grid.sel(dataset=cams_dataset, time=slice(tsel1, tsel2))
    if len(ds_sel.dataset) > 1:  # several datasets with same species
        # plot only if we have data in our given time period:
        variables_to_keep = [
            spec.dataset.values
            for spec in ds_sel.dataset
            if ds_sel.sel(dataset=spec)["value"].isnull().all() == False
        ]
        for ii in ds_sel.sel(dataset=variables_to_keep).dataset:
            plot_bias(cams["value"] - ds_sel.sel(dataset=ii)["value"])
    else:
        plot_bias(cams["value"] - ds_sel.sel(dataset=s)["value"])

    mbias = mean_bias(cams["value"], ds_sel.sel(dataset=s)["value"]).values
    rmse = root_mean_squared_error(cams["value"], ds_sel.sel(dataset=s)["value"]).values
    ax.text(
        1.05,
        0.2,
        f"Mean bias = {mbias:.2f} {ds_sel.unit[0].values} \nRMSE = {rmse:.2f} {ds_sel.unit[0].values}",
        horizontalalignment="left",
        verticalalignment="center",
        transform=ax.transAxes,
        fontsize='small'
    )


    ax.set_ylabel(f"{ds_sel.species[0].values} ({ds_sel.unit[0].values})")
    ax.set_xlabel("")
    ax.set_title(f"CAMS: {cams_dataset}")
    # ax.legend()
plt.suptitle("Bias (CAMS-obs)")
plt.tight_layout()

In [ ]:
## Figure for single species
data_sel = 'CO2'
cams_dataset = 'co2_invgg'

obs = ds_all_3h.sel(time=slice(tsel1, tsel2),dataset = data_sel, drop=True)
cams = cams_best_grid.sel(dataset=cams_dataset, time=slice(tsel1, tsel2))

fig, ax = plt.subplots(figsize=(6,4))
alpha=0.8
ms=8

pl_obs = obs['value'].plot(marker='.',ls='',label='Observations (3h mean)',alpha=alpha,markeredgewidth=0,markersize=ms)
pl_cams = cams['value'].plot(marker='.',ls='',label='CAMS',alpha=alpha,markeredgewidth=0,markersize=ms)
mw_days = 5 #moving window days
mw = 3*8*mw_days #  moving window
#cams_best.rolling(time=mw,center=True,min_periods=mw/2).mean().plot(ls='-',c='k')#c=pl_cams[0].get_color())
#obs3h.to_xarray().rolling(time=mw,center=True,min_periods=mw/2).mean().plot(ls='-',c=pl_obs[0].get_color())
plt.ylabel('CO2 (ppm)')
plt.xlabel('Time')
plt.legend()
plt.title('Mt. Kenya GAW station: \nCO2 observations and CAMS Model data')
form_xdate(ax,'Y',1,'')
fig_format='png'
plt.tight_layout()
#plt.savefig(fr'.\analyses\MKN_co2_gaw_cams_inv.{fig_format}',format=fig_format,dpi=400,transparent=False,facecolor='white')

#plt.savefig(f'.\\analyses\MKN_co2_gaw_cams_inv_mw{mw_days}d.{fig_format}',format=fig_format,dpi=400,transparent=False,facecolor='white')


### check GFAS data

In [ ]:
cams_gfas

In [ ]:
fig,ax = plt.subplots(1,1)
t1= '2020-01-01'
t2= '2023-12-31'
#t1= '2022-03-15'
#t2= '2022-03-31'
cams_gfas.mean(dim='latitude').mean(dim='longitude').sel(time=slice(t1,t2))['frpfire'].plot()
ax2 = ax.twinx()
#cams_gfas.mean(dim='latitude').mean(dim='longitude')['cofire'].sel(time=slice(t1,t2)).plot(ax=ax2,color='red')
plt.title('Wildfire radiative power in large area')
plt.tight_layout()
plt.show()

In [ ]:
tsel = '2022-03-15'
plt.figure()
cams_gfas.sel(time=tsel)['frpfire'].plot()
plt.show()

In [ ]:
gfas_sel

In [ ]:
tsel = "2022-03-15"
#gfas_sel = cams_gfas.sel(time=tsel)['frpfire']
# only show non-zero data
#fgfas_sel = gfas_sel.where(gfas_sel != 0, other=np.nan)

t1 = "2020-01-01"
t2 = "2023-12-31"
gfas_sel = cams_gfas["frpfire"].sel(time=slice(t1, t2)).mean(dim="time")
gfas_sel = gfas_sel.where(gfas_sel != 0, other=np.nan)

#area to map
[lon1,lon2,lat1,lat2]=[20,50,-10,20]
[lon1,lon2,lat1,lat2]=[0,60,-20,40]

fig = plt.figure()
ax = fig.add_subplot(1, 1, 1,projection=ccrs.PlateCarree())
plt.gca().set_extent([lon1,lon2,lat1,lat2],crs=ccrs.PlateCarree())

lands = cfeature.NaturalEarthFeature(category='physical', name='land',scale= '50m') #cfeature.COLORS['land']
ax.add_feature(lands,zorder=0)
ax.add_feature(cfeature.BORDERS, linestyle=":", edgecolor="black")

fire_log = -np.log10(gfas_sel)
ds_toplot = fire_log 
#ds_toplot = gfas_sel
plt.pcolormesh(
    gfas_sel["longitude"],
    gfas_sel["latitude"],
    ds_toplot,
    cmap="hot",
    transform=ccrs.PlateCarree(),
)
#or:
#gfas_sel.plot(ax=ax,cmap="hot",transform=ccrs.PlateCarree(),zorder=1)



cbar = plt.colorbar(label="-log(Fire Radiative Power [W/m^2])")
cbar.set_ticks(np.arange(np.floor(np.min(ds_toplot)), np.ceil(np.max(ds_toplot)) + 1))


mknlat, mknlon, mknalt = get_station_coords("MKN")
ax.plot(
    lon, lat, marker="o", color="green", transform=ccrs.PlateCarree(),
    zorder=2
)  # mt. kenya station

plt.show()

In [ ]:
## Mean fire vs. CO for different months
fig, ax = plt.subplots()
x = ds_all.sel(dataset="CO", time=slice(t1, t2))["value"].resample(time="D").mean()
y = cams_gfas.sel(time=slice(t1, t2)).mean(dim=["latitude", "longitude"])["frpfire"]
months = x.time.dt.month
# Define a colormap for months
colmap = tol_colors.tol_cmap('rainbow_discrete',12)# 12 colors for 12 months
scatter = ax.scatter(
    x,
    y,
    c=months,
    cmap=colmap,
)
plt.legend()
plt.xlabel('CO at MKN')
plt.ylabel('Mean fire activity (radiative power in W/m2)')
#plt.colorbar()
legend1 = ax.legend(*scatter.legend_elements(),
                    loc="upper right", title="Months")
ax.add_artist(legend1)


plt.show()

In [ ]:
cams_gfas.sel(time=slice(t1, t2))["dist_mkn"]

In [ ]:
# same but for north and south
mknlat, mknlon, mknalt = get_station_coords("MKN")
fig, axs = plt.subplots(1,2,figsize=(10,6),sharey=True,sharex=True)
## devide in different latitude regions (above and below the station)
for i,ax in enumerate(axs):
    y = ds_all.sel(dataset="CO", time=slice(t1, t2))["value"].resample(time="D").mean()

    # select fires above or below the station
    if i == 0:
        gfas_sel = cams_gfas.where(cams_gfas.latitude > mknlat)
        tit = 'Fire events north of MKN'
    else:
        gfas_sel = cams_gfas.where(cams_gfas.latitude < mknlat)
        tit = 'Fire events south of MKN'

    x = gfas_sel.sel(time=slice(t1, t2)).mean(dim=["latitude", "longitude"])["frpfire"]
    months = x.time.dt.month
    # Define a colormap for months
    colmap = tol_colors.tol_cmap("rainbow_discrete", 12)  # 12 colors for 12 months
    scatter = ax.scatter(
        x,
        y,
        c=months,
        cmap=colmap,
    )
    ax.set_xlabel("Mean fire activity (radiative power in W/m2)")
    ax.set_title(tit)
plt.legend()
axs[0].set_ylabel("CO at MKN")
# plt.colorbar()
legend1 = ax.legend(*scatter.legend_elements(), loc="upper right", title="Months")
ax.add_artist(legend1)
plt.suptitle(f'Mean regional fire events (CAMS) vs. CO at MKN between {t1} and {t2}')


plt.show()

In [ ]:
### Weight the fire activity with the distance to MKN station
dx = cams_gfas['longitude'] - mknlon
dy = cams_gfas['latitude'] - mknlat
dist = np.sqrt(dx**2+dy**2)
cams_gfas['dist_mkn'] = dist
fire_weighted = cams_gfas['frpfire'] * (1/dist)
cams_gfas['frpfire_weighted'] = fire_weighted

# TODO would need to make dist_mkn a coordinate, and make frpfire dependent on that, so that we have a dist value for each fire event!?

In [ ]:
## Mean fire vs. CO for different months
fig, ax = plt.subplots()
x = ds_all.sel(dataset="CO", time=slice(t1, t2))["value"].resample(time="D").mean()
y = cams_gfas.sel(time=slice(t1, t2)).mean(dim=["latitude", "longitude"])["frpfire"]
months = x.time.dt.month
# Define a colormap for months
colmap = tol_colors.tol_cmap('rainbow_discrete',12)# 12 colors for 12 months
area = cams_gfas.sel(time=slice(t1, t2)).mean(dim=["latitude", "longitude"])["dist_mkn"] ## TODO not working yet
scatter = ax.scatter(
    x,
    y,
    c=months,
    cmap=colmap,
    s=area
)
plt.legend()
plt.xlabel('CO at MKN')
plt.ylabel('Mean fire activity (radiative power in W/m2)')
#plt.colorbar()
legend1 = ax.legend(*scatter.legend_elements(),
                    loc="upper right", title="Months")
ax.add_artist(legend1)


plt.show()

In [ ]:
cams_gfas

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

d1 = "2022-03-20"
d2 = "2022-03-28"
var = "frpfire"
ds_sel = cams_gfas.sel(time=slice(d1, d2))
fig, axs = plt.subplots(3, 3, sharex=True, sharey=True, figsize=(8, 8))

for d, ax in zip(pd.date_range(start=d1, end=d2), axs.reshape(-1)):
    vmin = ds_sel[var].min()
    vmax = ds_sel[var].max()
    im = cams_gfas.sel(time=d)[var].plot(
        ax=ax, vmin=vmin, vmax=vmax, add_colorbar=False
    )
    ax.plot(lon, lat, marker="x", color="w")  # mt. kenya station
    ax.set_xlabel('lon')
    ax.set_ylabel('lat')
#common colorbar
fig.colorbar(
    im,
    ax=axs.ravel().tolist(),
    orientation="vertical",
    label="Wildfire radiative power (W/m2)",
    pad=-0.5,
)
plt.tight_layout()
plt.show()

In [ ]:
### Monthly anomalies

## plot cams versus observations
alpha = 0.7
ms = 8

fig, axs = plt.subplots(
    len(species_sel), 1, sharex=True, figsize=(10, len(species_sel) * 2)
)
for s, ax in zip(species_sel, axs):
    ds = ds_all #hourly observations

    ds_sel = ds.sel(time=slice(tsel1, tsel2)).where(ds.species == s, drop=True)
    # select cams datasets that exist for this species
    cams_datasets_sel = [cams_sel for cams_sel in cams_best_grid.dataset.values if f"{s.lower()}_" in cams_sel]
    cams = cams_best_grid.sel(dataset=cams_datasets_sel, time=slice(tsel1, tsel2))

    ## get monthly anomalies
    ds_m = get_anomalies(ds_sel.resample(time='MS').mean(),var='value',yr1=2020,yr2=2023)
    cams_m = get_anomalies(cams.resample(time='MS').mean(),var='value',yr1=2020,yr2=2023)

    # function to plot each subplot
    def plot_data(ds,label):
        if 'flask' in str(ds.dataset.values):
            m = '.'
            l = ''
        else:
            m = '.'
            l = '-'
        ds.plot(
            ax=ax,
            ls=l,
            marker=m,
            markeredgewidth=0,
            alpha=alpha,
            markersize=ms,
            label=f"{label} {str(ds.dataset.values)}",
        )

    # plot obs
    var_to_plot = "value_anom_per"
    if len(ds_m.dataset) > 1:  # several obs datasets with same species
        # plot only if we have data in our given time period:
        variables_to_keep = [
                spec.dataset.values
                for spec in ds_m.dataset
                if ds_m.sel(dataset=spec)[var_to_plot].isnull().all() == False
            ]
        for ds_i in ds_m.sel(dataset=variables_to_keep).dataset:
            plot_data(ds_m.sel(dataset=ds_i.values)["value_anom"], 'obs')
    else:
        plot_data(ds_m.sel(dataset=s)[var_to_plot], 'obs')

    # plot cams
    if len(cams_m.dataset) > 1:  # several cams datasets with same species
        # plot only if we have data in our given time period:
        variables_to_keep = [
                spec.dataset.values
                for spec in cams.dataset
                if cams_m.sel(dataset=spec)[var_to_plot].isnull().all() == False
            ]
        for cams_i in cams_m.sel(dataset=variables_to_keep).dataset:
            plot_data(cams_m.sel(dataset=cams_i)[var_to_plot], 'cams')
    else:
        plot_data(cams_m.isel(dataset=0)[var_to_plot], 'cams')

    ax.set_ylabel(f"{ds_sel.species[0].values} ({ds_sel.unit[0].values})")
    ax.set_xlabel("")
    ax.set_title("")
    ax.legend()
plt.suptitle('Monthly anomalies (data-climatology/climatology)')
plt.tight_layout()
plt.show()

### Check Seasonal and daily cycles

In [ ]:
## Remove some outliers, especially wild fire CO

import process_data
ds_all_3h_rem_out = process_data.rem_out(ds_all_3h,std_fac=10, z_threshold=4)

## plot removed data
var = 'value'

for s in species_sel:

    f, axs = plt.subplots(2, 1, sharex=True)
    plt.suptitle("Removed outliers")
    ds_all_3h.sel(dataset=s)[var].plot(ls="", marker="o", ax=axs[0])
    ds_all_3h.sel(dataset=s)[var + "_unc"].plot(ls="", marker="o", ax=axs[1])
    # new data:
    ds_all_3h_rem_out.sel(dataset=s)[var].plot(ls="", marker=".", ax=axs[0])
    ds_all_3h_rem_out.sel(dataset=s)[var + "_unc"].plot(ls="", marker=".", ax=axs[1])
    plt.show()

In [ ]:
## plot  SEASONAL CYCLE and DIURNAL cams versus observations

alpha = 0.7
ms = 8

# dont use all cams-products
cams_mask =  ~np.isin(cams_best_grid['dataset'], 'co2_egg4','ch4_egg4')
cams_use = cams_best_grid.sel(dataset=cams_mask)
#Or: use all: 
cams_use = cams_best_grid

for freq in ['month','hour']:
    fig, axs = plt.subplots(
        len(species_sel), 1, sharex=True, figsize=(5, len(species_sel) * 2)
    )
    for s, ax in zip(species_sel, axs):
        if s == "CH4":
            ds = ds_all  # ch4 has 6h resolution
        else:
            ds = ds_all

        ds_sel = ds.sel(time=slice(tsel1, tsel2)).where(ds.species == s, drop=True)
        # select cams datasets that exist for this species
        cams_datasets_sel = [cams_sel for cams_sel in cams_use.dataset.values if f"{s.lower()}_" in cams_sel]
        cams = cams_use.sel(dataset=cams_datasets_sel, time=slice(tsel1, tsel2))

        # function to plot each subplot
        def plot_data(ds,label):
            ds.groupby(f"time.{freq}").mean().plot(
                ax=ax,
                ls='-',
                marker='.',
                markeredgewidth=0,
                alpha=alpha,
                markersize=ms,
                label=f"{label} {str(ds.dataset.values)}",
            )

        # plot obs
        if len(ds_sel.dataset) > 1:  # several obs datasets with same species
            # plot only if we have data in our given time period:
            variables_to_keep = [
                    spec.dataset.values
                    for spec in ds_sel.dataset
                    if ds_sel.sel(dataset=spec)["value"].isnull().all() == False
                ]
            for ds_i in ds_sel.sel(dataset=variables_to_keep).dataset:
                plot_data(ds_sel.sel(dataset=ds_i.values)["value"], 'obs')
        else:
            plot_data(ds_sel.sel(dataset=s)["value"], 'obs')

        # plot cams
        if len(cams.dataset) > 1:  # several cams datasets with same species
            # plot only if we have data in our given time period:
            variables_to_keep = [
                    spec.dataset.values
                    for spec in cams.dataset
                    if cams.sel(dataset=spec)["value"].isnull().all() == False
                ]
            for cams_i in cams.sel(dataset=variables_to_keep).dataset:
                plot_data(cams.sel(dataset=cams_i)["value"], 'cams')
        else:
            plot_data(cams.isel(dataset=0)["value"], 'cams')

        ax.set_ylabel(f"{ds_sel.species[0].values} ({ds_sel.unit[0].values})")
        ax.set_xlabel("")
        ax.set_title("")
        ax.legend()
    
    if freq == 'month':
        tit = 'Seasonal'
    elif freq == 'hour':
        tit = 'Diurnal'

    plt.suptitle(f"{tit} cycle")
    plt.tight_layout()
    plt.show()

##### try to remove trend?

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

ds = ds_all.sel(dataset='CO2')['value']
x = ds[~np.isnan(ds)]
#x = ds.time.value[~np.isnan(ds['value'].values)]
#y = ds['value'].values[~np.isnan(ds['value'].values)]

decomposition = seasonal_decompose(x, model='additive', period=365*24)
x


In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

ds = ds_all.sel(dataset='CO2')['value']
x = ds[~np.isnan(ds)]
#x = ds.time.value[~np.isnan(ds['value'].values)]
#y = ds['value'].values[~np.isnan(ds['value'].values)]

decomposition = seasonal_decompose(x, model='additive', period=365*24)
x


ts_freq = 365 * 24
decomposition = seasonal_decompose(
    x, model="additive", period=ts_freq, #extrapolate_trend=ts_freq
)  # Assuming hourly data
plt.figure()
decomposition.plot()
plt.show()

detrended_data = x - decomposition.trend  # remove trend

# plt.figure()
# detrended_data.plot()
# plt.show()


detrended_data += (
    decomposition.seasonal + decomposition.observed.mean()
)  # add the mean value as intercept and seasonality

fig, ax = plt.subplots()
x.plot(ax=ax, label="normal")
detrended_data.plot(ax=ax, label="detrended",ls=':')
plt.legend()
plt.show()

In [ ]:
from statsmodels.tsa.seasonal import STL
ds = ds_all.sel(dataset='CO2').drop('dataset')['value']

x = ds[~np.isnan(ds)].to_dataframe()

stl = STL(x, period=24*365)
res = stl.fit() # this takes 4min!
fig = res.plot()

In [ ]:
## do the same but with monthly means


from statsmodels.tsa.seasonal import STL
dsM = ds_all.sel(dataset='CO2').resample(time='MS').mean().drop('dataset')['value']

x2 = dsM[~np.isnan(dsM)].to_dataframe()
plt.figure()
stl2 = STL(x2, period=12)
res2 = stl2.fit() # this takes 4min!
fig = res2.plot()

In [ ]:

detrended_data = x2.value - res2.trend  # remove trend

# plt.figure()
# detrended_data.plot()
# plt.show()


detrended_data += (
    #res2.seasonal 
    +res2.observed.mean().values
)  # add the mean value as intercept and seasonality (?)

fig, ax = plt.subplots()
dsM.plot(ax=ax, label="normal")
detrended_data.plot(ax=ax, label="detrended",ls=':',marker='.')
plt.legend()
plt.show()

In [ ]:
plt.figure()
dsM.groupby("time.month").mean().plot(label='normal')
detrended_data.to_xarray().groupby("time.month").mean().plot(label='detrend')
plt.legend()
plt.show()

In [ ]:
ds = ds_all.sel(dataset='CO2')['value']
ds[~np.isnan(ds)].to_dataframe()

In [ ]:
## time series difference
# => this removes also the seasonality!
ds = ds_all.sel(dataset='CO2')['value']
diff = ds.diff(dim='time')
ds_mean = ds.mean(dim='time')
plt.figure()
ds.plot(label='simpel detrended')
(diff + ds_mean).plot(ls=":",label='detrended (diff)')
plt.show()

In [ ]:
import statsmodels.api as sm

Y = x.values
X = x.time.values
X = sm.add_constant(X)
model = sm.OLS(Y,X)
results = model.fit()

In [ ]:
## SARIMA

import statsmodels.api as sm
mod = sm.tsa.statespace.SARIMAX(x.values, trend='c', order=(1,1,1))

res = mod.fit(disp=False)
print(res.summary())

In [ ]:
## Check autocorrelation
data = x.values
data['ln'] = np.log(data)
data['D.ln'] = data['ln'].diff()

fig, axes = plt.subplots(1, 2, figsize=(15,4))

fig = sm.graphics.tsa.plot_acf(data.iloc[1:]['D.ln'], lags=40, ax=axes[0])
fig = sm.graphics.tsa.plot_pacf(data.iloc[1:]['D.ln'], lags=40, ax=axes[1])

In [ ]:
modis = xr.open_dataset(r"C:\Users\leob\Documents\Data_analyses\KADI\git-repos\firetracks\MOD14A1\MOD14A1.A2019241.h08v05.006.2019250005105.hdf", engine='netcdf4')